# Day 3 MEDIUM: Compare Algorithms & Cross-Validate
### SDA AI Bootcamp: Supervised Learning for Regression & Classification

**Time:** ~40 minutes
**Datasets:** Bike-Share Demand (regression) & Student Pass/Fail (classification)
**Goal:** reproduce today's Algorithm Comparison, Cross-Validation, and Overfitting slides with your own code and real numbers.

Continues from Easy same datasets, now with all 4 applicable algorithms per task. Cells marked **`# TODO`** are for you; a collapsed ** Solution** follows each one.


## 1. Setup & Recap

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score

try:
    df = pd.read_csv("https://raw.githubusercontent.com/justmarkham/DAT8/master/data/bikeshare.csv")
except Exception as e:
    print("Download failed:", e, "upload bikeshare.csv manually")
    # from google.colab import files
    # df = pd.read_csv(list(files.upload().keys())[0])

features = ["season", "holiday", "workingday", "weather", "temp", "atemp", "humidity", "windspeed"]
X, y = df[features], df["count"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

try:
    sdf = pd.read_csv("https://raw.githubusercontent.com/arunk13/MSDA-Assignments/master/IS607Fall2015/Assignment3/student-mat.csv", sep=";")
except Exception as e:
    print("Download failed:", e, "upload student-mat.csv manually")
    # from google.colab import files
    # sdf = pd.read_csv(list(files.upload().keys())[0], sep=";")

sdf["Pass"] = (sdf["G3"] >= 10).astype(int)
Xs = sdf.drop(columns=["G1", "G2", "G3", "Pass"])
ys = sdf["Pass"]
cat_cols = Xs.select_dtypes(include="object").columns.tolist()
Xs_enc = Xs.copy()
for c in cat_cols:
    Xs_enc[c] = LabelEncoder().fit_transform(Xs_enc[c])
Xs_train, Xs_test, ys_train, ys_test = train_test_split(Xs_enc, ys, test_size=0.2, random_state=42, stratify=ys)
scaler2 = StandardScaler()
Xs_train_s, Xs_test_s = scaler2.fit_transform(Xs_train), scaler2.transform(Xs_test)

print("Regression:", X_train_s.shape, X_test_s.shape)
print("Classification:", Xs_train_s.shape, Xs_test_s.shape)


Regression: (8708, 8) (2178, 8)
Classification: (316, 30) (79, 30)


## 2. Regression Compare 4 Algorithms

**`# TODO`** Fill in the dictionary with all 4 regressors, then run the loop that trains each and records RMSE/R².

In [4]:
# TODO: complete this dict LinearRegression, DecisionTreeRegressor(max_depth=8, random_state=42),
# RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42), KNeighborsRegressor(n_neighbors=7)
reg_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    "KNN": KNeighborsRegressor(n_neighbors=7),
}  # My manual setup to compare all 4 regression algorithms and analyze their errors

reg_results = []
for name, model in reg_models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2 = r2_score(y_test, preds)
    reg_results.append({"Algorithm": name, "RMSE": round(rmse, 2), "R2": round(r2, 3)})

reg_results_df = pd.DataFrame(reg_results).sort_values("RMSE")
reg_results_df


,Algorithm,RMSE,R2
2,Random Forest,143.87,0.373
3,KNN,148.52,0.332
1,Decision Tree,151.66,0.303
0,Linear Regression,154.62,0.276


<details><summary> Solution</summary>

```python
reg_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    "KNN": KNeighborsRegressor(n_neighbors=7),
}
```

Your table should closely match the lecture's Algorithm Comparison slide: Random Forest best (RMSE ≈ 143.9), then KNN, then Decision Tree, then Linear Regression.
</details>

In [5]:
# Self-check
assert len(reg_results_df) == 4
assert reg_results_df.iloc[0]["Algorithm"] == "Random Forest", "Expected Random Forest to have the lowest RMSE"
print("Regression comparison looks correct.")


Regression comparison looks correct.


## 3. Classification Compare 4 Algorithms

**`# TODO`** Same pattern, classification side. Don't be surprised if the "simplest" algorithm wins here that's a real result, not a mistake.

In [6]:
# TODO: complete this dict LogisticRegression(max_iter=2000), DecisionTreeClassifier(max_depth=5, random_state=42),
# RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42), KNeighborsClassifier(n_neighbors=7)
clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),  # Tested 4 classification models to compare Accuracy and F1-score performance
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7),
}

clf_results = []
for name, model in clf_models.items():
    model.fit(Xs_train_s, ys_train)
    preds = model.predict(Xs_test_s)
    acc = accuracy_score(ys_test, preds)
    f1 = f1_score(ys_test, preds)
    clf_results.append({"Algorithm": name, "Accuracy": round(acc, 3), "F1": round(f1, 3)})

clf_results_df = pd.DataFrame(clf_results).sort_values("Accuracy", ascending=False)
clf_results_df


,Algorithm,Accuracy,F1
0,Logistic Regression,0.684,0.783
1,Decision Tree,0.658,0.769
2,Random Forest,0.646,0.770
3,KNN,0.646,0.767


<details><summary> Solution</summary>

```python
clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7),
}
```

Logistic Regression should come out on top here (Accuracy ≈ 0.684) matching the lecture's honest finding that Random Forest does NOT always win.
</details>

In [7]:
# Self-check
assert len(clf_results_df) == 4
assert clf_results_df.iloc[0]["Algorithm"] == "Logistic Regression", "Expected Logistic Regression to have the highest accuracy on this dataset"
print("Classification comparison looks correct.")


Classification comparison looks correct.


## 4. Cross-Validation Is That Score Trustworthy?

A single train/test split could have gotten lucky or unlucky. **`# TODO`** Run 5-fold cross-validation on the Random Forest classifier and look at the spread.

In [8]:
# TODO: run cross_val_score with cv=5, scoring="accuracy", on the FULL (unsplit) Xs_enc/ys
cv_scores = cross_val_score(RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
    Xs_enc, ys, cv=5, scoring="accuracy") # 5-fold CV to test Random Forest performance
print("Fold scores:", np.round(cv_scores, 3))
print("Mean:", cv_scores.mean().round(3), " Std Dev:", cv_scores.std().round(3))


Fold scores: [0.696 0.747 0.684 0.633 0.62 ]
Mean: 0.676  Std Dev: 0.046


<details><summary> Solution</summary>

```python
cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
    Xs_enc, ys, cv=5, scoring="accuracy"
)
```

Your fold scores should closely match the lecture's k-fold slide: roughly [0.696, 0.747, 0.684, 0.633, 0.620], mean ≈ 0.676. Notice the single split we used above (0.646) sits within that spread, but isn't the same as the average exactly why we don't trust one split alone.
</details>

## 5. Overfitting in Action

**`# TODO`** Train Decision Trees at a few different `max_depth` values and watch train vs. test accuracy diverge.

In [10]:
# TODO: for each depth in [1, 3, 5, 8, 10, None], fit a DecisionTreeClassifier(max_depth=depth, random_state=42)
# on Xs_train_s and record train accuracy vs test accuracy

# Testing overfitting by changing tree depths
depth_results = []
for depth in [1, 3, 5, 8, 10, None]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(Xs_train_s, ys_train)
    train_acc = accuracy_score(ys_train, dt.predict(Xs_train_s))
    test_acc = accuracy_score(ys_test, dt.predict(Xs_test_s))
    depth_results.append({"max_depth": str(depth), "train_acc": round(train_acc,3), "test_acc": round(test_acc,3)})

pd.DataFrame(depth_results)


,max_depth,train_acc,test_acc
0,1,0.737,0.671
1,3,0.769,0.658
2,5,0.829,0.658
3,8,0.908,0.709
4,10,0.943,0.633
5,None,1.000,0.658


<details><summary> Solution</summary>

The loop above is already complete this section is about reading the result, not writing new code. You should see train accuracy climb toward 100% as depth increases, while test accuracy plateaus around depth=8 and doesn't keep improving the same overfitting curve from the lecture slide, reproduced with your own run.
</details>

---
## Reflection

1. Which algorithm won on which task? Was it the same algorithm both times?
2. Look at the cross-validation fold scores if you'd only run the single split from Section 3, would you have drawn the same conclusion about this model's quality?
3. At what `max_depth` did the test accuracy peak? What happened to the train/test gap after that point?


 [ 1- No it wasn't the same simpler models or different algorithms performed best depending on the specific regression and classification metrics

 2- Not necessary because a single split can be affected by luck good or bad whereas cross-validation gives a more reliable and trustworthy average score

 3- he test accuracy peaked at moderate depth like 5 and after that point the gap between train and test accuracy widened which is a clear sign of overfitting ]
---
### Next up
**Hard** adds proper hyperparameter tuning with `GridSearchCV`, a tuned-vs-untuned comparison, and an independent data-leakage challenge.

### Dataset credit
Bike-share dataset via `justmarkham/DAT8`. Student Performance dataset (Cortez & Silva, 2008) via UCI Machine Learning Repository.
